# 🚀 Hardware-Accelerated WebGPU Chrome on Google Colab (Tesla T4)
این نوت‌بوک محیط سرور لینوکس ابری گوگل کولب را برای اجرای مرورگر Google Chrome با شتاب کامل سخت‌افزاری WebGPU روی کارت گرافیک **NVIDIA Tesla T4** آماده و استریم می‌کند.

### نحوه استفاده:
1. مطمئن شوید از منوی **Runtime -> Change runtime type** گزینه **T4 GPU** انتخاب شده باشد.
2. سلول اول را اجرا (Play) کنید.
3. پس از حدود ۵۰ ثانیه روی دکمه سبز رنگ "ورود به مرورگر کروم ریموت" کلیک کنید.


In [ ]:
#@title 🚀 1-Click Fast WebGPU Chrome Launcher for Google Colab
#@markdown اجرای همه‌چیز در یک مرحله (پیکربندی خودکار درایورهای Vulkan انویدیا، پکیج‌ها، کروم و ساخت لینک ریموت پرسرعت در کمتر از ۶۰ ثانیه)

TARGET_URL = "https://unicred.fun/#mine" #@param {type:"string"}
RESOLUTION = "1280x720" #@param ["1280x720", "1600x900", "1920x1080"]

import os, sys, time, subprocess, re

print("⏳ [1/6] پیکربندی درایورهای گرافیکی و دسترسی سخت‌افزاری به NVIDIA T4...")
os.makedirs("/dev/dri", exist_ok=True)
subprocess.run("mknod -m 666 /dev/dri/card0 c 226 0 2>/dev/null", shell=True)
subprocess.run("mknod -m 666 /dev/dri/renderD128 c 226 128 2>/dev/null", shell=True)
subprocess.run("mknod -m 666 /dev/nvidia-modeset c 195 254 2>/dev/null", shell=True)

os.makedirs("/etc/vulkan/icd.d", exist_ok=True)
os.makedirs("/usr/share/vulkan/icd.d", exist_ok=True)
icd_content = '{\n    "file_format_version": "1.0.0",\n    "ICD": {\n        "library_path": "libGLX_nvidia.so.0",\n        "api_version": "1.3.275"\n    }\n}'
for p in ["/etc/vulkan/icd.d/nvidia_icd.json", "/usr/share/vulkan/icd.d/nvidia_icd.json"]:
    with open(p, "w") as f: f.write(icd_content)

with open("/etc/ld.so.conf.d/nvidia.conf", "w") as f:
    f.write("/usr/lib64-nvidia\n")
subprocess.run("ldconfig", shell=True)

print("⏳ [2/6] نصب پکیج‌های ریموت دسکتاپ و کروم...")
subprocess.run("apt-get update -qq && apt-get install -y -qq vulkan-tools libvulkan1 xvfb fluxbox x11vnc websockify novnc autocutsel xclip > /dev/null 2>&1", shell=True)

if not os.path.exists("/usr/bin/google-chrome"):
    print("⏳ [3/6] دانلود و نصب Google Chrome Stable...")
    subprocess.run("wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb && dpkg -i google-chrome-stable_current_amd64.deb > /dev/null 2>&1 || apt-get install -fy -qq > /dev/null 2>&1 && rm -f google-chrome-stable_current_amd64.deb", shell=True)
else:
    print("⚡ [3/6] کروم از قبل نصب است.")

if not os.path.exists("/usr/local/bin/cloudflared"):
    print("⏳ [4/6] دانلود تونل Cloudflare...")
    subprocess.run("wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared", shell=True)

print("⏳ [5/6] راه‌اندازی استک دسکتاپ بهینه‌شده...")
subprocess.run("pkill -9 -f 'Xvfb|x11vnc|websockify|fluxbox|cloudflared|google-chrome'", shell=True)
time.sleep(1)

env = os.environ.copy()
env["DISPLAY"] = ":1"
env["VK_ICD_FILENAMES"] = "/etc/vulkan/icd.d/nvidia_icd.json"

subprocess.Popen(["Xvfb", ":1", "-screen", "0", f"{RESOLUTION}x24"], env=env)
time.sleep(1)
subprocess.Popen(["fluxbox"], env=env)
subprocess.Popen(["autocutsel", "-fork", "-s", "CLIPBOARD"], env=env)
subprocess.Popen(["autocutsel", "-fork", "-s", "PRIMARY"], env=env)
time.sleep(1)

subprocess.Popen(["x11vnc", "-display", ":1", "-nopw", "-listen", "127.0.0.1", "-forever", "-rfbport", "5900", "-wait", "5", "-defer", "5", "-threads", "-nowf", "-noxrecord"], env=env)
time.sleep(1)
subprocess.Popen(["websockify", "--web", "/usr/share/novnc", "6080", "localhost:5900"], env=env)
time.sleep(1)

print("⏳ [6/6] اجرای کروم با شتاب کامل سخت‌افزاری WebGPU روی تسلا T4...")
chrome_cmd = [
    "google-chrome",
    "--no-sandbox",
    "--disable-gpu-sandbox",
    "--enable-unsafe-webgpu",
    "--enable-features=Vulkan,DefaultANGLEVulkan,VulkanFromANGLE,WebGPU",
    "--use-vulkan=native",
    "--use-angle=vulkan",
    "--ignore-gpu-blocklist",
    "--disable-dev-shm-usage",
    "--enable-gpu-rasterization",
    "--enable-zero-copy",
    "--disable-background-timer-throttling",
    "--disable-backgrounding-occluded-windows",
    "--disable-renderer-backgrounding",
    f"--window-size={RESOLUTION.split('x')[0]},{RESOLUTION.split('x')[1]}",
    "--start-maximized",
    "--user-data-dir=/tmp/chrome_colab_profile",
    TARGET_URL
]
subprocess.Popen(chrome_cmd, env=env)
time.sleep(2)

log_file = "/tmp/cloudflared.log"
if os.path.exists(log_file): os.remove(log_file)
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:6080", "--logfile", log_file])

tunnel_url = None
for _ in range(25):
    time.sleep(1)
    if os.path.exists(log_file):
        with open(log_file) as f:
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if m:
                tunnel_url = m.group(0)
                break

if tunnel_url:
    link = f"{tunnel_url}/vnc.html?autoconnect=true&resize=scale&quality=6&compression=2"
    from IPython.display import display, HTML
    print("\n" + "═"*60)
    print("✅ همه‌چیز آماده است! (NVIDIA Tesla T4 + Hardware WebGPU)")
    print("═"*60)
    display(HTML(f'<br><a href="{link}" target="_blank" style="padding:14px 28px; background:#00c853; color:#000; font-weight:bold; font-size:18px; text-decoration:none; border-radius:8px; display:inline-block;">👉 ورود به مرورگر کروم ریموت</a><br><br><b>لینک مستقیم:</b> {link}'))
    print("═"*60)
else:
    print("❌ خطا در دریافت آدرس تونل. لطفاً دوباره سلول را اجرا کنید.")


In [ ]:
#@title 📊 Real-Time GPU & Hashrate Monitor
import time, subprocess
from IPython.display import clear_output

print("Starting lightweight GPU monitor (press Stop button in notebook to halt)...")
try:
    while True:
        res = subprocess.run([
            "nvidia-smi",
            "--query-gpu=name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu,power.draw",
            "--format=csv,noheader,nounits"
        ], capture_output=True, text=True)
        parts = [p.strip() for p in res.stdout.strip().split(",")]
        if len(parts) >= 7:
            name, gpu_util, mem_util, mem_used, mem_total, temp, power = parts
            clear_output(wait=True)
            print("="*60)
            print(f"🚀 UNICRED WEBGPU MINING MONITOR — {name}")
            print("="*60)
            print(f"  GPU Compute Utilization : {gpu_util}%")
            print(f"  VRAM Memory Used        : {mem_used} MiB / {mem_total} MiB ({mem_util}%)")
            print(f"  GPU Temperature         : {temp} °C")
            print(f"  Power Consumption       : {power} W")
            print("="*60)
        time.sleep(1)
except KeyboardInterrupt:
    print("Monitor stopped.")
